In [1]:
import pandas as pd
import numpy as np

In [2]:
# Loading the data

df = pd.read_csv(r"C:\Users\Shayan Mukherjee\OneDrive\Documents\Campaign messy data.csv")

print(f"Loaded Dataset: {df.shape[0]}-rows, {df.shape[1]}-columns")

Loaded Dataset: 2020-rows, 12-columns


In [13]:
# Step 1 : Data inspection :

print(df.info())
print(df.describe(include='all'))
print(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2020 entries, 0 to 2019
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   campaign_id    2020 non-null   object        
 1   campaign_name  2020 non-null   object        
 2   start_date     1687 non-null   datetime64[ns]
 3   end_date       2020 non-null   datetime64[ns]
 4   channel        2020 non-null   object        
 5   impressions    2020 non-null   int64         
 6   clicks         2020 non-null   int64         
 7   spend          2020 non-null   float64       
 8   conversions    2020 non-null   float64       
 9   active         2020 non-null   bool          
 10  campaign_tag   2020 non-null   object        
dtypes: bool(1), datetime64[ns](2), float64(2), int64(2), object(4)
memory usage: 159.9+ KB
None
       campaign_id        campaign_name                     start_date  \
count         2020                 2020                    

In [20]:
# Step 2: Checking for Duplicates:

print("Duplicate Rows:", df.duplicated().sum())

df = df.drop_duplicates()

print('Changes Applied')

print("Duplicate Rows:", df.duplicated().sum())

Duplicate Rows: 19
Changes Applied
Duplicate Rows: 0


In [4]:
# Step 3 : Handle Missing Values

print("Missing Values Before Cleaning:")
print(df.isnull().sum())

# Remove unnecessary column
df = df.drop(columns=['Clicks2'])

# Fill channel
df['Channel'] = df['Channel'].fillna('Unknown')

# Fill conversions
df['Conversions'] = df['Conversions'].fillna(0)

print("Missing Values After Cleaning:")
print(df.isnull().sum())

Missing Values Before Cleaning:
Campaign_ID         0
Campaign_Name       0
Start_Date          0
End_Date            0
Channel           101
Impressions         0
Clicks              0
Spend               0
Conversions       200
Active              0
Clicks2          1980
Campaign_Tag        0
dtype: int64
Missing Values After Cleaning:
Campaign_ID      0
Campaign_Name    0
Start_Date       0
End_Date         0
Channel          0
Impressions      0
Clicks           0
Spend            0
Conversions      0
Active           0
Campaign_Tag     0
dtype: int64


In [5]:
# Step 4 : Cleaning Header

print(df.columns.tolist())

df.columns = df.columns.str.strip().str.lower().str.replace(' ','_')

print ("Changes Applied")
print(df.columns.tolist())

['Campaign_ID', 'Campaign_Name', 'Start_Date', 'End_Date', 'Channel', 'Impressions', 'Clicks', 'Spend', 'Conversions', 'Active', 'Campaign_Tag']
Changes Applied
['campaign_id', 'campaign_name', 'start_date', 'end_date', 'channel', 'impressions', 'clicks', 'spend', 'conversions', 'active', 'campaign_tag']


In [6]:
# Step 5 : Type Conversion and Currency cleaning

print(df['spend'])

df['spend'] = pd.to_numeric(df['spend'].replace(r'[^0-9.-]', '', regex=True))

print("Fix Applied")
print (df['spend'])

0       $102.82
1         24.33
2       1323.39
3       2180.38
4        252.44
         ...   
2015    $503.95
2016       1641
2017     883.82
2018     4198.5
2019    1315.59
Name: spend, Length: 2020, dtype: object
Fix Applied
0        102.82
1         24.33
2       1323.39
3       2180.38
4        252.44
         ...   
2015     503.95
2016    1641.00
2017     883.82
2018    4198.50
2019    1315.59
Name: spend, Length: 2020, dtype: float64


In [7]:
# Step 6 : Categorical Typing error

print(df['channel'].unique())

clean_channel = {
    'Facebok':'Facebook',
    'Insta_gram':'Instagram',
    'Tik_Tok':'TikTok',
    'E-mail':'Email',
    'Gogle':'Google Ads',
}

df['channel'] = df['channel'].replace(clean_channel)

print('Changes Applied')
print(df['channel'].unique())

['TikTok' 'Facebook' 'Email' 'Instagram' 'Google Ads' 'E-mail' 'Unknown'
 'Gogle' 'Tik_Tok' 'Facebok' 'Insta_gram']
Changes Applied
['TikTok' 'Facebook' 'Email' 'Instagram' 'Google Ads' 'Unknown']


In [8]:
# Step 7 : Handling the Boolean values in the data

print(df['active'].unique())

clean_bool = {
    'Y': True,
    'Yes': True,
    '1': True,
    'TRUE': True,
    1: True,
    'No': False,
    '0': False,
    'FALSE': False,
    0: False
}

df['active'] = df['active'].map(clean_bool)

print('Fix Applied')
print(df['active'].unique())

['Y' '0' 'No' 'TRUE' 'Yes' '1' 'FALSE']
Fix Applied
[ True False]


In [9]:
# Step 8 : Date Parsing

print(df['start_date'].dtype)

df['start_date'] = pd.to_datetime(df['start_date'], dayfirst=True, errors = 'coerce')
df['end_date'] = pd.to_datetime(df['end_date'], dayfirst=True,  errors = 'coerce')

print('Fix Applied')
print(df['start_date'].dtype)
print(df['end_date'].dtype)

object
Fix Applied
datetime64[ns]
datetime64[ns]


In [10]:
# Step 9 : Logical Thinking ( Clicks vs Impressions)

irregular = df['clicks'] > df['impressions']
print(df.loc[irregular,['campaign_id','clicks','impressions']].head())

Empty DataFrame
Columns: [campaign_id, clicks, impressions]
Index: []


In [11]:
# Step 10 : Logical Integrity (Time)

time_gap = df['end_date'] < df['start_date']

print(df.loc[time_gap,['campaign_id','end_date','start_date']].head(5))

# Business assumption:
# Campaigns are assumed to run for 30 days.

df.loc[time_gap,'end_date'] = df.loc[time_gap,'start_date'] + pd.Timedelta(days=30)

print('Fix Applied')
print(df.loc[time_gap,['campaign_id','end_date','start_date']].head(5))

    campaign_id   end_date start_date
23    CMP-00024 2023-05-01 2023-05-06
54    CMP-00055 2023-08-27 2023-09-01
71    CMP-00072 2023-01-27 2023-02-01
156   CMP-00157 2023-12-01 2023-12-06
200   CMP-00201 2023-01-06 2023-01-11
Fix Applied
    campaign_id   end_date start_date
23    CMP-00024 2023-06-05 2023-05-06
54    CMP-00055 2023-10-01 2023-09-01
71    CMP-00072 2023-03-03 2023-02-01
156   CMP-00157 2024-01-05 2023-12-06
200   CMP-00201 2023-02-10 2023-01-11


In [12]:
# Step 11 : Handle Outliers

Q1 = df['spend'].quantile(0.25)
Q3 = df['spend'].quantile(0.75)

IQR = Q3 - Q1

# Using 3 × IQR to identify only extreme outliers

upper = Q3 + (3 * IQR)

outlier = df['spend'] > upper

print(df.loc[outlier,['campaign_id','spend']].head(5))

print('Fix Applied')
df.loc[outlier,'spend'] = upper
print(df.loc[outlier,['campaign_id','spend']].head(5))

     campaign_id      spend
789    CMP-00790  500000.00
1443   CMP-01444    8921.51
1460   CMP-01461  500000.00
1718   CMP-01719  500000.00
1754   CMP-01755  500000.00
Fix Applied
     campaign_id      spend
789    CMP-00790  8603.5375
1443   CMP-01444  8603.5375
1460   CMP-01461  8603.5375
1718   CMP-01719  8603.5375
1754   CMP-01755  8603.5375


In [27]:
# Step 12 : Final Report ⭐⭐⭐⭐⭐

print("=" * 60)
print("FINAL DATA CLEANING REPORT")
print("=" * 60)

print(f"Rows              : {df.shape[0]}")
print(f"Columns           : {df.shape[1]}")
print(f"Missing Values    : {df.isnull().sum().sum()}")
print(f"Duplicate Rows    : {df.duplicated().sum()}")

print("\nData Types")
print(df.dtypes)

print("\nDataset is ready for analysis.")

FINAL DATA CLEANING REPORT
Rows              : 1669
Columns           : 11
Missing Values    : 0
Duplicate Rows    : 0

Data Types
campaign_id              object
campaign_name            object
start_date       datetime64[ns]
end_date         datetime64[ns]
channel                  object
impressions               int64
clicks                    int64
spend                   float64
conversions             float64
active                     bool
campaign_tag             object
dtype: object

Dataset is ready for analysis.


In [28]:
# Step 13 : Clean Data set 

df.to_csv(
    "Marketing_Campaign_Cleaned.csv",
    index=False
)

print("Cleaned dataset exported successfully.")

Cleaned dataset exported successfully.
